# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
  OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
  Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1580, Days: 16184


In [3]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

In [ ]:
# === RELOADING FROM PARQUET ===
features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

In [ ]:
# tickers = ['NVDA', 'GOOG']
# tickers = ["NVDA"]
tickers = ["GOOG"]
df = df_ohlcv.loc[tickers]
# print(df.head())
# print('===========\n')
print(f"df.info():\n{df.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 5453 entries, ('GOOG', Timestamp('2004-08-19 00:00:00')) to ('GOOG', Timestamp('2026-04-22 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Open   5453 non-null   float64
 1   Adj High   5453 non-null   float64
 2   Adj Low    5453 non-null   float64
 3   Adj Close  5453 non-null   float64
 4   Volume     5453 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 953.8+ KB
df.info():
None


In [ ]:
# print(f"df_indices.info():\n{df_indices.info()}")
# print(f"{'='*20}\n")
# print(f"df_indices.index.names:\n{df_indices.index.names}")
# print(f"{'='*20}\n")
# print(f"df_ohlcv.info():\n{df_ohlcv.info()}")
# print(f"{'='*20}\n")
# print(f"df_ohlcv.index.names:\n{df_ohlcv.index.names}")
# print(f"{'='*20}\n")
# print(f"df_HY_spread_T10Y2Y_spread.info():\n{df_HY_spread_T10Y2Y_spread.info()}")
# print(f"{'='*20}\n")
# print(
#     f"df_HY_spread_T10Y2Y_spread.index.names:\n{df_HY_spread_T10Y2Y_spread.index.names}"
# )

In [ ]:
# print(f"features_df.info():\n{features_df.info()}\n")
# print(f"{'='*20}\n")
# print(f"features_df.index.names:\n{features_df.index.names}\n")
# print(f"{'='*20}\n")
# print(f"macro_df.info():\n{macro_df.info()}\n")
# print(f"{'='*20}\n")
# print(f"macro_df.index.names:\n{macro_df.index.names}\n")

## III. The Analysis Suite

In [13]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [ ]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2027-01-01"),
    lookback_period=189,
    holding_period=5,
    metric="Oversold (-RSI)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=100,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

⚠️ DATA/UI MISMATCH WARNING
Requested Decision Date: 2027-01-01 is not available.
The UI Decision Date input box is showing a date beyond available history.
REPLACING WITH LATEST AVAILABLE DATE: 2026-04-14


DEBUG: 942 stocks passed filters on 2026-04-14


In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

## IV. Deep Dives & Verification

In [ ]:
def verify_alpha_integrity(engine: AlphaEngine, ticker: str, date: pd.Timestamp):
    """Manual audit of the 4 New Pillars (Autocorr, Range, OBV, Convexity)."""
    alpha_matrix = engine.compute_alpha_matrix(decision_date=date, lookback_period=21)
    if ticker not in alpha_matrix.index:
        return print("❌ Ticker filtered out.")

    # --- Autocorr ---
    engine_val = alpha_matrix.loc[ticker, "Return Autocorr (15d)"]
    raw_rets = (
        engine.df_ohlcv_raw.xs(ticker, level="Ticker")["Adj Close"]
        .pct_change()
        .loc[:date]
        .tail(16)
    )
    manual_val = raw_rets.corr(raw_rets.shift(1))
    print(
        f"Autocorr (15d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    # --- Range ---
    engine_val = alpha_matrix.loc[ticker, "Range Position (20d)"]
    t_df = engine.df_ohlcv_raw.xs(ticker, level="Ticker").loc[:date].tail(20)
    manual_val = (t_df["Adj Close"].iloc[-1] - t_df["Adj Low"].min()) / (
        t_df["Adj High"].max() - t_df["Adj Low"].min()
    )
    print(
        f"Range Pos (20d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    print("🎯 Audit Complete.")


verify_alpha_integrity(engine, "NVDA", pd.Timestamp("2026-03-31"))

In [ ]:
@dataclass
class FourDRegime:
    regime: str
    signal: str
    confidence: float
    risk_flags: list
    raw_values: dict
    metadata: dict


def analyze_4d_regime(engine, ticker, date, mode="hybrid"):
    """Logic for classifying market state based on 4 pillars."""
    # ... Implementation details ... (Truncated for readability, keep original logic in file)
    pass  # Replace with full original function during reorganization execution


# result = analyze_4d_regime(engine, "NVDA", pd.Timestamp("2026-03-31"))

## V. Builders & Heavy Lifting

In [ ]:
# --- THE MARATHON BAKE ---
CHECKPOINT_DIR = OUTPUT_DIR / "alpha_cache_v1_checkpoints"
FINAL_FILE = OUTPUT_DIR / "AlphaCache_Master_2000-2026.parquet"

# ParallelFeatureBuilder.run_marathon(
#     master_engine=engine,
#     lookbacks=[10, 21, 63, 189],
#     start_date="2000-01-01",
#     checkpoint_dir=CHECKPOINT_DIR,
#     batch_size=50,
#     num_workers=max(1, multiprocessing.cpu_count() - 2),
# )

# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

## VI. Forensic Sandbox & Utilities

In [ ]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

In [ ]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [15]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 3))
[  4]   📂 tickers (len=3)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]   📈 initial_weights (shape=(3,))
[  9]   📂 perf_metrics (len=24)
[ 10]     🔢 full_p_gain (float)
[ 11]     🔢 full_p_sharpe (float)
[ 12]     🔢 full_p_sharpe_atrp (float)
[ 13]     🔢 full_p_sharpe_trp (float)
[ 14]     🔢 lookback_p_gain (float)
[ 15]     🔢 lookback_p_sharpe (float)
[ 16]     🔢 lookback_p_sharpe_atrp (float)
[ 17]     🔢 lookback_p_sharpe_trp (float)
[ 18]     🔢 holding_p_gain (float)
[ 19]     🔢 holding_p_sharpe (float)
[ 20]     🔢 holding_p_sharpe_atrp (float)
[ 21]     🔢 holding_p_sharpe_trp (float)
[ 22]     🔢 full_b_gain (float)
[ 23]     🔢 full_b_sharpe (float)
[ 24]     🔢 full_b_sharpe_atrp (float)
[ 25]     🔢 full_b_sharpe_trp (float)
[ 26]     🔢 lookback_b_gain (flo

In [22]:
# # SU.peek(50, result)
# SU.peek(50, result).info()
# df_snapshot = SU.peek(50, result)
df_snapshot.to_csv(OUTPUT_DIR / "df_snapshot.csv")

In [37]:
print(SU.peek(49, result))

 📍 INDEX: [49]
 🏷️  NAME:  tickers_passed
 📂 PATH:  audit_pack -> debug_data -> audit_liquidity -> tickers_passed



942

942


In [ ]:
# for i in [35, 36, 37, 38]:
for i in [5, 6, 7]:
    print(SU.peek(i, result))
    print("*********")

 📍 INDEX: [5]
 🏷️  NAME:  index_0
 📂 PATH:  audit_pack -> tickers -> index_0



'HOLX'

HOLX
*********
 📍 INDEX: [6]
 🏷️  NAME:  index_1
 📂 PATH:  audit_pack -> tickers -> index_1



'GIS'

GIS
*********
 📍 INDEX: [7]
 🏷️  NAME:  index_2
 📂 PATH:  audit_pack -> tickers -> index_2



'CAG'

CAG
*********


In [ ]:
SU.peek(1, result)

 📍 INDEX: [1]
 🏷️  NAME:  portfolio_series
 📂 PATH:  audit_pack -> portfolio_series



Date
2026-03-30    1.0000
2026-03-31    1.0010
2026-04-01    0.9967
2026-04-02    1.0030
2026-04-06    1.0081
2026-04-07    0.9942
2026-04-08    0.9943
2026-04-09    0.9950
2026-04-10    0.6468
2026-04-13    0.6231
2026-04-14    0.6196
2026-04-15    0.6136
2026-04-16    0.6341
2026-04-17    0.6392
2026-04-20    0.6367
2026-04-21    0.6310
2026-04-22    0.6293
dtype: float64

Date
2026-03-30    1.0000
2026-03-31    1.0010
2026-04-01    0.9967
2026-04-02    1.0030
2026-04-06    1.0081
2026-04-07    0.9942
2026-04-08    0.9943
2026-04-09    0.9950
2026-04-10    0.6468
2026-04-13    0.6231
2026-04-14    0.6196
2026-04-15    0.6136
2026-04-16    0.6341
2026-04-17    0.6392
2026-04-20    0.6367
2026-04-21    0.6310
2026-04-22    0.6293
dtype: float64

In [24]:
# SU.peek(51, result)
# SU.peek(51, result).info()
df_full_universe_ranking = SU.peek(51, result)
df_full_universe_ranking.to_csv(OUTPUT_DIR / "df_full_universe_ranking.csv")

 📍 INDEX: [51]
 🏷️  NAME:  full_universe_ranking
 📂 PATH:  audit_pack -> debug_data -> full_universe_ranking



,Strategy_Score,Raw_Price_Start,Raw_Price_End,Raw_TRP_Mean,Raw_ATRP_Mean,Raw_Mom_21,Raw_IR_63,Raw_Consistency,Raw_RSI,Raw_DD_21,Raw_Range_Pos_20,Raw_Slope_P_5,Raw_Slope_V_5,Raw_Convexity,Raw_AutoCorr_15,Was_Dropped
Ticker,,,,,,,,,,,,,,,,
A,-60.4093,111.755,120.390,0.0270,0.0276,0.0821,-0.2277,0.6,60.4093,0.0000,0.8972,0.0077,5.2036e+05,0.0060,-0.1776,False
AA,-62.0850,63.220,71.840,0.0474,0.0530,0.1297,0.0583,0.4,62.0850,-0.0201,0.8130,0.0003,4.9378e+05,-0.0058,0.1554,False
AAL,-60.1185,10.180,12.130,0.0530,0.0499,0.1777,-0.1379,0.4,60.1185,0.0000,0.9148,0.0110,-1.2794e+07,-0.0016,-0.1149,False
AAPL,-53.0674,246.630,258.830,0.0232,0.0227,0.0348,-0.0017,0.4,53.0674,-0.0064,0.7986,-0.0006,-2.4308e+07,-0.0045,-0.2210,False
ABBV,-45.6746,211.366,208.530,0.0280,0.0272,-0.0429,-0.0328,0.6,45.6746,-0.0505,0.3978,-0.0041,-1.5268e+06,-0.0082,0.1787,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZBH,-67.3938,88.380,96.520,0.0224,0.0239,0.0384,0.0610,1.0,67.3938,0.0000,0.8944,0.0099,1.9390e+06,0.0035,-0.0526,False
ZBRA,-59.9110,199.300,226.700,0.0391,0.0404,0.1183,-0.0840,0.8,59.9110,0.0000,0.8598,0.0063,3.7570e+05,-0.0087,-0.2147,False
ZM,-54.1349,78.680,82.400,0.0401,0.0356,0.1120,-0.0143,0.4,54.1349,-0.0194,0.6581,-0.0035,-1.5515e+05,0.0050,-0.5300,False


In [ ]:
start_date = "2026-03-30"
end_date = "2026-04-22"

idx = pd.IndexSlice
df_subset = features_df.loc[idx[:, start_date:end_date], :]
# subset.head(20)
df_subset.to_csv(OUTPUT_DIR / "df_subset.csv")

In [26]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9525692 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-04-22 00:00:00'))
Data columns (total 18 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Consistency          float64
 6   IR_63                float64
 7   Beta_63              float64
 8   DD_21                float64
 9   AutoCorr_15          float64
 10  Ret_1d               float64
 11  Range_Pos_20         float64
 12  Slope_P_5            float64
 13  Slope_V_5            float64
 14  Convexity            float64
 15  RollingStalePct      float64
 16  RollMedDollarVol     float64
 17  RollingSameVolCount  float64
dtypes: float64(18)
memory usage: 1.3+ GB
